# 第10回　回帰分析：単回帰と重回帰
## ―― データを「数式（モデル）」で説明する

統計学Ⅰ（B）　／　北星学園大学

注目は ――

> モデルは現実の**近似**にすぎない。**当てはまりの良さ（R²）は、良いモデルの証明ではない。**

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats
import statsmodels.api as sm

def _build_from_source():
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    SRC = "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt"
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = pd.read_csv(SRC, sep="\t")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    return out.replace(-999, np.nan).sort_values("学名").reset_index(drop=True)

try:
    df = pd.read_csv("https://aonoa68.github.io/toukei-1/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## 0. まず、うまくいかない例から

**行動圏の広さ**を予測したい。手始めに `栄養段階`（1＝草食、2＝雑食）で予測してみる。

> **今日も対数で扱う。** 体重も行動圏も何桁にもまたがるので、第3回・第5回と同じ理由である。

In [ ]:
e = df.dropna(subset=["行動圏km2", "栄養段階"]).copy()
e = e[e["行動圏km2"] > 0]
y = np.log10(e["行動圏km2"])

X = sm.add_constant(e[["栄養段階"]])
m0 = sm.OLS(y, X).fit()
print(f"傾き = {m0.params['栄養段階']:.4f}　R² = {m0.rsquared:.4f}   (n={len(e)}種)")
print("→ R²はほぼ0。草食か雑食かでは、行動圏の広さをまったく説明できない。")

何も説明できなかった（R² ≈ 0.0001）。では **体重** ならどうか？

In [ ]:
s = df.dropna(subset=["行動圏km2", "体重g"]).copy()
s = s[(s["行動圏km2"] > 0) & (s["体重g"] > 0)]
s["log行動圏"] = np.log10(s["行動圏km2"])
s["log体重"]   = np.log10(s["体重g"])

m1 = sm.OLS(s["log行動圏"], sm.add_constant(s[["log体重"]])).fit()
切片 = m1.params["const"]; 傾き = m1.params["log体重"]

print(f"log行動圏 ≈ {切片:.3f} + {傾き:.3f} × log体重")
print(f"R² = {m1.rsquared:.3f}（行動圏のばらつきの約{m1.rsquared*100:.0f}%を説明）  n={len(s)}種")

plt.figure(figsize=(6.5,4.5))
plt.scatter(s["log体重"], s["log行動圏"], s=16, alpha=0.5, color="#00897b")
xs = np.linspace(s["log体重"].min(), s["log体重"].max(), 50)
plt.plot(xs, 切片 + 傾き*xs, color="#e8503a", lw=2, label="回帰直線")
plt.xticks([2,3,4,5], ["100g","1kg","10kg","100kg"])
plt.yticks([-2,-1,0,1], ["0.01km²","0.1km²","1km²","10km²"])
plt.xlabel("体重（対数）"); plt.ylabel("行動圏（対数）"); plt.legend()
plt.title(f"単回帰：R² = {m1.rsquared:.2f}"); plt.show()

---
## 1. 回帰の言葉

- **回帰直線**：データに最もよく当てはまる直線（最小二乗法：縦のズレの二乗和が最小）
- **傾き（回帰係数）**：説明変数が1増えると、予測値がいくつ増えるか
- **切片**：説明変数が0のときの予測値
- **決定係数 R²**（0〜1）：目的変数のばらつきのうち、モデルが説明できた割合

### 両対数のときの傾きは「何倍になるか」を表す

今回の傾きは **0.933**。両方を対数にしているので、これは足し算ではなく**かけ算**の関係を意味する。

In [ ]:
print(f"傾き = {傾き:.3f}")
print()
for 倍 in [2, 10, 100]:
    print(f"  体重が {倍:>3}倍 になると、行動圏は {倍**傾き:>5.1f}倍 と予測される")
print()
print("（傾きが1.0ちょうどなら『体重10倍で行動圏も10倍』。")
print("  0.933 はそれよりわずかに小さい ＝ 大きい種ほど体重あたりの行動圏はやや狭い）")

この傾き（**スケーリング指数**）は、生物学では意味のある量として扱われる。「体が大きいほど広い範囲が要る」ことは分かりきっているが、**どういう比率で増えるのか**を1つの数で表せるのが回帰の力である。

> ⚠️ ただし「体重を増やせば行動圏が広がる」という**因果の保証ではない**（第5回）。> 回帰係数は**予測上の関連**を表しているだけである。

---
## 2. 重回帰 ―― 複数の説明変数で

体重だけでなく、**集団サイズ**（何頭で暮らすか）も一緒に使う。重回帰では各係数が「**他の変数を一定にしたときの**、その変数の関連」を表す（＝統制）。

In [ ]:
t = df.dropna(subset=["行動圏km2", "体重g", "集団サイズ"]).copy()
t = t[(t[["行動圏km2","体重g","集団サイズ"]] > 0).all(axis=1)]
for col in ["行動圏km2", "体重g", "集団サイズ"]:
    t["log" + col] = np.log10(t[col])

m2 = sm.OLS(t["log行動圏km2"], sm.add_constant(t[["log体重g", "log集団サイズ"]])).fit()
print(f"log行動圏 ≈ {m2.params['const']:.3f} + {m2.params['log体重g']:.3f}×log体重 "
      f"+ {m2.params['log集団サイズ']:.3f}×log集団サイズ")
print(f"R² = {m2.rsquared:.3f}   n={len(t)}種   （単回帰では {m1.rsquared:.3f} だった）")
print()
print("係数の符号：どちらも正。体が大きいほど、また大きな群れほど、広い範囲を使う。")

**R² が 0.46 → 0.60 に上がった。** 集団サイズには、体重だけでは説明できない情報がある。

注目してほしいのは **体重の係数が 0.933 → 0.568 に下がった**ことである。「集団サイズが同じ種どうしで比べたときの、体重の効果」はもっと小さい。単回帰のときの0.933には、**集団サイズの効果が混ざり込んでいた**（大きい種は大きな群れを作りがち）。

> これが第5回でやった**統制**そのものである。重回帰は、統制を式の中で行う道具でもある。

In [ ]:
# さらに個体群密度（1km²あたり何頭いるか）を足す
u = df.dropna(subset=["行動圏km2", "体重g", "集団サイズ", "個体群密度"]).copy()
u = u[(u[["行動圏km2","体重g","集団サイズ","個体群密度"]] > 0).all(axis=1)]
for col in ["行動圏km2", "体重g", "集団サイズ", "個体群密度"]:
    u["log" + col] = np.log10(u[col])

m3 = sm.OLS(u["log行動圏km2"],
            sm.add_constant(u[["log体重g", "log集団サイズ", "log個体群密度"]])).fit()
print(f"R² = {m3.rsquared:.3f}   n={len(u)}種")
print()
for k, v in m3.params.items():
    print(f"  {k:<14} {v:+.3f}")
print()
print("個体群密度の係数が負：密に住んでいる種ほど、一頭あたりの行動圏は狭い。")

R² は **0.75** まで上がった。よく当てはまっている ―― ように見える。

**ここで立ち止まる。** 変数を足したから上がったのか、それとも本当に良くなったのか。

---
## 3. R²の罠 ―― 変数を足せば、必ず上がる

ここが今日の核心。**説明変数を足すと、R²は必ず上がる（決して下がらない）。**たとえそれが**まったく無意味な数字**でも。

確かめよう：行動圏とは何の関係もない**でたらめな乱数の列**を足していく。

In [ ]:
rng = np.random.default_rng(2026)
base = s[["log体重"]].copy()

print("『体重』に、行動圏と無関係な乱数列を足していくと R² は…")
for k in [0, 1, 5, 20, 50]:
    X = base.copy()
    for j in range(k):
        X[f"でたらめ{j}"] = rng.normal(0, 1, len(s))   # 意味のない数字
    r2 = sm.OLS(s["log行動圏"], sm.add_constant(X)).fit().rsquared
    print(f"  でたらめ列 {k:2d}本追加: R² = {r2:.3f}")
print()
print("中身が無意味でも、変数を足すほど R² は上がる。")
print("→ 『R²が高い＝良いモデル』ではない！ 変数を増やせばいくらでも盛れる。")

でたらめな列を50本足すだけで、R²は **0.46 → 0.67** に上がった。中身はゴミなのに「当てはまり」は良くなる。これが **過学習** の入り口だ。

しかも今回は n=153 種しかない。**データの数に対して変数が多いほど、この効き方は激しくなる。**

> §2で R² が 0.75 まで上がったのを見て「良いモデルだ」と思ったなら、> いま同じことが起きていないと、どうやって言えるだろうか。

> 当てはまり（R²）を上げたいだけなら、変数をたくさん入れればいい。だが、それは**手元のデータに合わせすぎた**だけで、**未来の予測は良くならない**。
> 「当てはまりの良さ」と「モデルの複雑さ」をどう釣り合わせるか ―― それが**次回（第11回 AIC）**のテーマ。

---
## 4. 残差を見る習慣

**残差**＝実際の値 − 予測値。残差を予測値に対してプロットし、偏った模様がないか確認するのが作法（ランダムに散っていれば良い兆候）。

In [ ]:
plt.figure(figsize=(6.5,3.8))
plt.scatter(m1.fittedvalues, m1.resid, s=16, alpha=0.5, color="#00897b")
plt.axhline(0, color="#e8503a", lw=2)
plt.xlabel("予測値（log行動圏）"); plt.ylabel("残差（実際−予測）")
plt.title("残差プロット：偏りなく散っていればOK")
plt.show()

print("予測より行動圏がはるかに広い3種：")
out = s.assign(残差=m1.resid).nlargest(3, "残差")[["学名", "体重g", "行動圏km2", "残差"]]
for _, r in out.iterrows():
    print(f"  {r['学名']:<26} {r['体重g']:>9,.0f}g  {r['行動圏km2']:>6.1f}km²  "
          f"予測の {10**r['残差']:>4.0f}倍")

### 残差は「外れ」ではなく「発見」でもある

予測から最も外れているのは **Erythrocebus patas（パタスモンキー）**。8kgの体格から予測される行動圏の **約49倍** を使っている。

これはデータの誤りだろうか。**そうではない。**パタスモンキーは森林ではなく**サバンナで地上生活する**霊長類で、散在する食物を求めて広大な範囲を移動することが知られている。

> **残差の大きい点は、モデルが捉えていない何かを教えてくれる。**
> 「当てはまりが悪い」で終わらせず、**なぜ外れたのかを見に行く**。
> そこにこそ、平均的な傾向では見えない現象がある。

---
## 5. 外挿の危険 ―― 範囲の外は予測できない

In [ ]:
print(f"モデルを当てはめた体重の範囲: {s['体重g'].min():.0f}g 〜 {s['体重g'].max():,.0f}g")
print()
for w, ラベル in [(1_000_000, "1トンの霊長類"), (1, "1gの霊長類"), (3_000_000, "アフリカゾウ（3トン）")]:
    pred = 10 ** (切片 + 傾き * np.log10(w))
    print(f"  体重 {w:>9,} g を式に入れると → 行動圏 {pred:>9,.4f} km²   ← {ラベル}")

**式はどんな数字を入れても答えを返す。** エラーは出ない。

しかし ―― **1トンの霊長類も、1gの霊長類も存在しない。**そしてアフリカゾウは霊長類ですらないので、霊長類153種から作った式を当てる理由がない。

> **モデルはデータのある範囲でしか信用できない。**
> ここが第2回・第3回の外れ値の話と違うところに注意してほしい。> 外れ値は「ありえない値かどうか」を人間が判断できた。> **外挿は、出てきた数字が正しいかどうか確かめる相手がそもそも存在しない。**

---
## 今日のまとめ

| 概念 | ひとこと |
|---|---|
| 単回帰 | y ≈ 切片 + 傾き×x。傾き＝xが1増えると予測がいくつ増えるか |
| 両対数の傾き | 「何倍になるか」を表す（体重10倍 → 行動圏8.6倍） |
| 決定係数 R² | ばらつきのうちモデルが説明できた割合（0〜1） |
| 重回帰 | 複数の説明変数。各係数は「他を一定にしたとき」の関連（＝統制） |
| ❌ R²の罠 | 変数を足せばR²は必ず上がる。**でたらめ50本で0.46→0.67** |
| ❌ 係数の罠 | 回帰係数＝予測上の関連。因果の保証ではない |
| ❌ 外挿の罠 | データ範囲の外は予測できない（1トンの霊長類はいない） |
| 残差 | 実際−予測。偏りを見る。**大きな残差は発見の入り口**（パタスモンキー） |

> **モデルは現実の近似。「当てはまりの良さ」は「良いモデル」ではない。**
> 係数を因果と読まず、範囲外を予測せず、残差を見る。

**課題（Moodle）**：回帰結果を解釈し、「この予測の限界・使ってはいけない場面」を述べる。

---

!!! quote "このデータの出典"
    Jones, K.E. et al. (2009) PanTHERIA: a species-level database of life history,
    ecology, and geography of extant and recently extinct mammals.
    *Ecology* 90(9): 2648. Ecological Archives E090-184.

    霊長類376種の行だけを抜き出し、列を選び、気温の単位を直したもの。値は変えていない。